# Week 1: Data Curation and Preprocessing


In [1]:
#environment setup 
%pip install -q pandas datasets transformers indic-nlp-library


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
#importing all required libraries
import json
import platform
import re
import unicodedata
import requests
from importlib.metadata import version
from pathlib import Path
from io import StringIO

import pandas as pd
from datasets import load_dataset
from IPython.display import display

#checking if the loading has done correctly or not
for package in ["pandas", "datasets", "transformers", "indic-nlp-library"]:
    print(package, version(package))

# structure of repo
week1_dir = Path("Week1") if Path("Week1").is_dir() else Path(".")
output_dir = week1_dir / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

pandas 2.3.3
datasets 5.1.0
transformers 5.19.0
indic-nlp-library 0.92


In [8]:
# IndicSentiment: validation and test only
repo = "ai4bharat/IndicSentiment"
revision = "dc8f3f66886531c6897fedffca1e938a68fc5013"
languages = ["hi", "bn", "mr"]

frames = []

for split in ["validation", "test"]:
    for lang in languages:
        url = (
            f"https://huggingface.co/datasets/{repo}/resolve/"
            f"{revision}/data/{split}/{lang}.json"
        )

        response = requests.get(url, timeout=60)
        response.raise_for_status()

        # Support JSON Lines and regular JSON
        try:
            df = pd.read_json(StringIO(response.text), lines=True)
        except ValueError:
            df = pd.DataFrame(response.json())

        df["language"] = lang
        df["split"] = split
        frames.append(df)

sentiment = pd.concat(frames, ignore_index=True)

assert {"INDIC REVIEW", "LABEL"} <= set(sentiment.columns), (
    f"Unexpected columns: {sentiment.columns.tolist()}"
)

sentiment["text_raw"] = sentiment["INDIC REVIEW"]
sentiment["label"] = sentiment["LABEL"]

print("Sentiment shape:", sentiment.shape)
print("\nRows by split and language:")
print(sentiment.groupby(["split", "language"]).size())

display(sentiment.head())

Sentiment shape: (3468, 14)

Rows by split and language:
split       language
test        bn          1000
            hi          1000
            mr          1000
validation  bn           156
            hi           156
            mr           156
dtype: int64


,GENERIC CATEGORIES,CATEGORY,SUB-CATEGORY,PRODUCT,BRAND,ASPECTS,ASPECT COMBO,ENGLISH REVIEW,LABEL,INDIC REVIEW,language,split,text_raw,label
0,Home,Appliances,Home theater,Soundbars,Boat,"Bluetooth/wireless, HDMI, audio output mode, i...",HDMI,This boat's soundbar is still wire-connectivit...,Negative,बोट के साउंडबार में अभी भी सभी स्पीकर्स के लिए...,hi,validation,बोट के साउंडबार में अभी भी सभी स्पीकर्स के लिए...,Negative
1,Hobbies,Music,Audio Output,headphones,Zeb Paradise,"on-ear, in-ear, wired, bluetooth, earbuds, noi...",Over-ear with mic,Foldable type of microphone with mic and micro...,Positive,माइक और माइक्रो एसडी कार्ड स्लॉट के साथ फोल्डे...,hi,validation,माइक और माइक्रो एसडी कार्ड स्लॉट के साथ फोल्डे...,Positive
2,Entertainment,Apps,Social Media,Social networking,Instagram,"find friends, share photos and moments, free m...",daily status,The recently included feature of stories by de...,Positive,हाल ही में डाला गया फीचर जो डिफ़ॉल्ट रूप से 24...,hi,validation,हाल ही में डाला गया फीचर जो डिफ़ॉल्ट रूप से 24...,Positive
3,Transportation,Air,Flights,International,Emirates,"luggage allowance, affordable rates, luxury, f...",Rates Luggage allowance,"Rates are competitive, almost always the best ...",Positive,"""रेट्स प्रतिस्पर्धी हैं, लगभग मार्किट में सबसे...",hi,validation,"""रेट्स प्रतिस्पर्धी हैं, लगभग मार्किट में सबसे...",Positive
4,Home,Appliances,Fan,Exhaust fan,Bajaj Maxima,"remove moisture/unpleasant odour, air delivery...",Front Shutter,Looks very big and efficient. But since there ...,Negative,बहुत बड़ा और बेहतरीन दिखता है। लेकिन चूंकि कोई...,hi,validation,बहुत बड़ा और बेहतरीन दिखता है। लेकिन चूंकि कोई...,Negative


In [11]:
# Samanantar(not full corpus)
samanantar_repo = "ai4bharat/samanantar"
parallel_limit = 5_000

frames = []
for lang in languages:
    stream = load_dataset(samanantar_repo, lang, split="train", streaming=True)

    rows = []
    for row in stream:
        rows.append(row)
        if len(rows) >= parallel_limit:
            break

    df = pd.DataFrame(rows)
    df["language"] = lang
    frames.append(df)
    print(f"{lang}: loaded {len(df):,} sentence pairs")

parallel = pd.concat(frames, ignore_index=True)

# src = English, tgt = Indic translation
parallel["english_raw"] = parallel["src"]
parallel["indic_raw"] = parallel["tgt"]
print("Parallel shape:", parallel.shape)

hi: loaded 5,000 sentence pairs
bn: loaded 5,000 sentence pairs
mr: loaded 5,000 sentence pairs
Parallel shape: (15000, 6)


In [12]:
# data exploration 
print("Columns:", list(sentiment.columns))
display(pd.crosstab(sentiment["language"], sentiment["split"]))
display(sentiment["label"].value_counts(dropna=False).to_frame("rows"))
display(sentiment.isna().sum().to_frame("missing"))
print("Intent column present:", any("intent" in col.lower() for col in sentiment.columns))
display(sentiment.groupby("language").head(2)[["language", "text_raw", "label"]])

Columns: ['GENERIC CATEGORIES', 'CATEGORY', 'SUB-CATEGORY', 'PRODUCT', 'BRAND', 'ASPECTS', 'ASPECT COMBO', 'ENGLISH REVIEW', 'LABEL', 'INDIC REVIEW', 'language', 'split', 'text_raw', 'label']


split,test,validation
language,,
bn,1000,156
hi,1000,156
mr,1000,156


,rows
label,
Positive,1743
Negative,1719
None,6


,missing
GENERIC CATEGORIES,0
CATEGORY,0
SUB-CATEGORY,0
PRODUCT,0
BRAND,24
ASPECTS,0
ASPECT COMBO,21
ENGLISH REVIEW,0
LABEL,6
INDIC REVIEW,0


Intent column present: False


,language,text_raw,label
0,hi,बोट के साउंडबार में अभी भी सभी स्पीकर्स के लिए...,Negative
1,hi,माइक और माइक्रो एसडी कार्ड स्लॉट के साथ फोल्डे...,Positive
156,bn,এই বোটের সাউন্ডবারটি এখনও সব স্পিকারের জন্য তা...,Negative
157,bn,মাইক ও মাইক্রো এসডি কার্ড স্লটসহ ভাঁজযোগ্য ধরন...,Positive
312,mr,बोटच्या या साऊंडबारमध्ये अजूनही सर्व स्पीकर्सस...,Negative
313,mr,माईकसह घडी करता येणारा मायक्रोफोन आणि मायक्रो ...,Positive


In [15]:
# Parallel text
print("Columns:", list(parallel.columns))
display(parallel.groupby("language").size().to_frame("pairs"))
display(parallel.groupby("language").head(2)[["language", "english_raw", "indic_raw"]])

Columns: ['idx', 'src', 'tgt', 'language', 'english_raw', 'indic_raw']


,pairs
language,
bn,5000
hi,5000
mr,5000


,language,english_raw,indic_raw
0,hi,"However, Paes, who was partnering Australia's ...",आस्ट्रेलिया के पाल हेनली के साथ जोड़ी बनाने वाल...
1,hi,"Whosoever desires the reward of the world, wit...",और जो शख्स (अपने आमाल का) बदला दुनिया ही में च...
5000,bn,The International Civil Aviation Organisation ...,আন্তর্জাতিক সিভিল এভিয়েশন সংস্থা বিমানবন্দরের ...
5001,bn,They can be deactivated using the control pane...,আপনার বাহুতে থাকা কন্ট্রোল প্যানেল দ্বারা ঐ জু...
10000,mr,"Today, ASEAN and India enjoy multi-faceted coo...","आज आसियान आणि भारत, आसियानचे राजनीतिक- सुरक्षा..."
10001,mr,"For us in India, ASEANs continuance in this ro...",आसियान राष्ट्र समुहातील देशांशी भारताचे उत्तम ...


In [16]:
# Cleaning the text (URLs, e-mails, @mentions and HTML-like tags are removed)
def clean_text(text):
    if not isinstance(text, str):
        return ""

    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+\.\S+", " ", text)
    text = re.sub(r"(?<!\S)@[A-Za-z0-9_.]+", " ", text)
    text = re.sub(r"#([^\s#]+)", lambda m: m.group(1).replace("_", " "), text)
    text = re.sub(r"<[^>]+>", " ", text)

    # Remove invisible formatting characters, but preserve ZWJ/ZWNJ
    text = text.translate(dict.fromkeys(map(ord, "\u200b\u200e\u200f\u00ad\ufeff")))

    # Normalize digits to ASCII
    text = "".join(
        str(unicodedata.decimal(ch))
        if unicodedata.category(ch) == "Nd" else ch
        for ch in text
    )

    # Reduce exaggerated repetitions, e.g. "बहुतtttt" -> "बहुतttt"
    text = re.sub(r"([^\s\d])\1{3,}", lambda m: m.group(1) * 3, text)

    # Normalize extra whitespace
    return re.sub(r"\s+", " ", text).strip()

In [ ]:
# testing the function on a few examples
examples = [
    "यह अच्छा है 😚 #बहुत_अच्छा https://example.com @user",
    "দারুণ পণ্য!!!!!! mail me: a@b.com ১২৩",
    "हा फोन छान आहे 👍 www.test.in #छान",
    "क\u200dष और क\u200cष",
]

for example in examples:
    print("Before:", example)
    print("After: ", clean_text(example), "\n")

assert "\u200d" in clean_text(examples[3]) and "\u200c" in clean_text(examples[3])

Before: यह अच्छा है 😚 #बहुत_अच्छा https://example.com @user
After:  यह अच्छा है 😚 बहुत अच्छा 

Before: দারুণ পণ্য!!!!!! mail me: a@b.com ১২৩
After:  দারুণ পণ্য!!! mail me: 123 

Before: हा फोन छान आहे 👍 www.test.in #छान
After:  हा फोन छान आहे 👍 छान 

Before: क‍ष और क‌ष
After:  क‍ष और क‌ष 



In [18]:
# applying cleaning
sentiment["text"] = sentiment["text_raw"].apply(clean_text)
parallel["english_text"] = parallel["english_raw"].apply(clean_text)
parallel["indic_text"] = parallel["indic_raw"].apply(clean_text)

print("Rows changed by cleaning:")
print("sentiment:", (sentiment["text_raw"].fillna("") != sentiment["text"]).sum())
print("parallel (Indic side):", (parallel["indic_raw"].fillna("") != parallel["indic_text"]).sum())

display(sentiment.groupby("language").head(3)[["language", "text_raw", "text"]])

Rows changed by cleaning:
sentiment: 1204
parallel (Indic side): 3324


,language,text_raw,text
0,hi,बोट के साउंडबार में अभी भी सभी स्पीकर्स के लिए...,बोट के साउंडबार में अभी भी सभी स्पीकर्स के लिए...
1,hi,माइक और माइक्रो एसडी कार्ड स्लॉट के साथ फोल्डे...,माइक और माइक्रो एसडी कार्ड स्लॉट के साथ फोल्डे...
2,hi,हाल ही में डाला गया फीचर जो डिफ़ॉल्ट रूप से 24...,हाल ही में डाला गया फीचर जो डिफ़ॉल्ट रूप से 24...
156,bn,এই বোটের সাউন্ডবারটি এখনও সব স্পিকারের জন্য তা...,এই বোটের সাউন্ডবারটি এখনও সব স্পিকারের জন্য তা...
157,bn,মাইক ও মাইক্রো এসডি কার্ড স্লটসহ ভাঁজযোগ্য ধরন...,মাইক ও মাইক্রো এসডি কার্ড স্লটসহ ভাঁজযোগ্য ধরন...
158,bn,24 ঘণ্টার জন্য গল্পের যে বৈশিষ্ট্যটি ডিফল্টভাব...,24 ঘণ্টার জন্য গল্পের যে বৈশিষ্ট্যটি ডিফল্টভাব...
312,mr,बोटच्या या साऊंडबारमध्ये अजूनही सर्व स्पीकर्सस...,बोटच्या या साऊंडबारमध्ये अजूनही सर्व स्पीकर्सस...
313,mr,माईकसह घडी करता येणारा मायक्रोफोन आणि मायक्रो ...,माईकसह घडी करता येणारा मायक्रोफोन आणि मायक्रो ...
314,mr,अलीकडेच समाविष्ट केलेल्या स्टोरीज ह्या 24 तासा...,अलीकडेच समाविष्ट केलेल्या स्टोरीज ह्या 24 तासा...


In [19]:
from indicnlp.tokenize import indic_tokenize

# Tokenizing
def tokenize(text, lang):
    return indic_tokenize.trivial_tokenize(text, lang)

Compare the two outputs above: whitespace splitting leaves punctuation such as the danda `।` attached to words. Tokenization alone does not align the languages; model tokenizers (for example SentencePiece-based) are chosen in Week 2.

In [21]:
# Applying tokenization
sentiment["tokens"] = sentiment.apply(lambda row: tokenize(row["text"], row["language"]), axis=1)
parallel["indic_tokens"] = parallel.apply(lambda row: tokenize(row["indic_text"], row["language"]), axis=1)

print("Average tokens per text:")
print("sentiment:", round(sentiment["tokens"].apply(len).mean(), 2))
print("parallel (Indic side):", round(parallel["indic_tokens"].apply(len).mean(), 2))

display(sentiment.groupby("language").head(2)[["language", "text", "tokens"]])

Average tokens per text:
sentiment: 25.71
parallel (Indic side): 14.41


,language,text,tokens
0,hi,बोट के साउंडबार में अभी भी सभी स्पीकर्स के लिए...,"[बोट, के, साउंडबार, में, अभी, भी, सभी, स्पीकर्..."
1,hi,माइक और माइक्रो एसडी कार्ड स्लॉट के साथ फोल्डे...,"[माइक, और, माइक्रो, एसडी, कार्ड, स्लॉट, के, सा..."
156,bn,এই বোটের সাউন্ডবারটি এখনও সব স্পিকারের জন্য তা...,"[এই, বোটের, সাউন্ডবারটি, এখনও, সব, স্পিকারের, ..."
157,bn,মাইক ও মাইক্রো এসডি কার্ড স্লটসহ ভাঁজযোগ্য ধরন...,"[মাইক, ও, মাইক্রো, এসডি, কার্ড, স্লটসহ, ভাঁজযো..."
312,mr,बोटच्या या साऊंडबारमध्ये अजूनही सर्व स्पीकर्सस...,"[बोटच्या, या, साऊंडबारमध्ये, अजूनही, सर्व, स्प..."
313,mr,माईकसह घडी करता येणारा मायक्रोफोन आणि मायक्रो ...,"[माईकसह, घडी, करता, येणारा, मायक्रोफोन, आणि, म..."


In [23]:
# language check
script_ranges = {"hi": (0x0900, 0x097F), "mr": (0x0900, 0x097F), "bn": (0x0980, 0x09FF)}

def script_share(text, lang):
    letters = [ch for ch in text if ch.isalpha()]
    if not letters:
        return float("nan")
    low, high = script_ranges[lang]
    return sum(low <= ord(ch) <= high for ch in letters) / len(letters)

def quality_report(df, name, known_labels=None):
    print(f"--- {name} ---")
    print("Missing raw text:", df["text_raw"].isna().sum())
    print("Empty cleaned text:", df["text"].str.strip().eq("").sum())
    print("Missing labels:", df["label"].isna().sum())
    if known_labels:
        print("Unexpected labels:", (~df["label"].dropna().astype(str).str.lower().isin(known_labels)).sum())
    print("Duplicate texts within language and split:", df.duplicated(subset=["language", "split", "text"]).sum())
    shares = df.apply(lambda row: script_share(row["text"], row["language"]), axis=1)
    print("Under 50% letters in the expected script:", (shares < 0.5).sum())

quality_report(sentiment, "Sentiment", ["positive", "negative", "neutral"])


--- Sentiment ---
Missing raw text: 0
Empty cleaned text: 0
Missing labels: 6
Unexpected labels: 0
Duplicate texts within language and split: 6
Under 50% letters in the expected script: 0


In [25]:
# Split overlap and parallel-pair checks
for name, df, first, second in [("sentiment", sentiment, "validation", "test")]:
    overlap = (set(df[df["split"] == first]["text"]) & set(df[df["split"] == second]["text"])) - {""}
    print(f"{name}: texts in both {first} and {second}:", len(overlap))

print("\n--- Parallel (Samanantar sample) ---")
print("Empty English sentences:", parallel["english_text"].eq("").sum())
print("Empty Indic sentences:", parallel["indic_text"].eq("").sum())
print("Duplicate pairs:", parallel.duplicated(subset=["language", "english_text", "indic_text"]).sum())
shares = parallel.apply(lambda row: script_share(row["indic_text"], row["language"]), axis=1)
print("Indic sentences under 50% letters in the expected script:", (shares < 0.5).sum())

print("\nSentiment labels by language and split:")
display(pd.crosstab([sentiment["language"], sentiment["split"]], sentiment["label"]))

sentiment: texts in both validation and test: 0

--- Parallel (Samanantar sample) ---
Empty English sentences: 0
Empty Indic sentences: 0
Duplicate pairs: 0
Indic sentences under 50% letters in the expected script: 42

Sentiment labels by language and split:


label                Negative  Positive
language split                         
bn       test             492       506
         validation        81        75
hi       test             492       506
         validation        81        75
mr       test             492       506
         validation        81        75

In [26]:
#saving the processed data
outputs = {
    "indic_sentiment_processed.csv": sentiment[["language", "split", "text_raw", "text", "label", "tokens"]],
    "samanantar_parallel_sample.csv": parallel[["language", "english_raw", "indic_raw", "english_text", "indic_text"]],
}

for file_name, frame in outputs.items():
    frame = frame.copy()
    if "tokens" in frame.columns:
        frame["tokens"] = frame["tokens"].apply(lambda tokens: json.dumps(tokens, ensure_ascii=False))

    path = output_dir / file_name
    frame.to_csv(path, index=False, encoding="utf-8")
    print("Saved to:", path)

    group_cols = [col for col in ["language", "split"] if col in frame.columns]
    display(frame.groupby(group_cols).size().to_frame("rows"))

Saved to: data/processed/indic_sentiment_processed.csv


rows
language split           
bn       test        1000
         validation   156
hi       test        1000
         validation   156
mr       test        1000
         validation   156

Saved to: data/processed/samanantar_parallel_sample.csv


,rows
language,
bn,5000
hi,5000
mr,5000
